In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [5]:
def load_data(file_path: str):
    """
    Loads the safrin03 dataset
    """
    # 1. Load the data defensively
    df = pd.read_csv(file_path)
    return df


In [6]:
def transform_columns(df: pd.DataFrame):
    clean_cols = df.columns.str.strip().str.replace(' ', '_')

    # Insert underscore between lowercase letters/digits and an uppercase letter (e.g., CustomerID -> Customer_ID)
    clean_cols = clean_cols.str.replace(r'([a-z0-9])([A-Z])', r'\1_\2', regex=True)

    # Insert underscore between consecutive uppercase letters followed by lowercase (e.g., USAUser -> USA_User)
    clean_cols = clean_cols.str.replace(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', regex=True)

    # Lowercase everything and collapse any sequential underscores (e.g., customer__id -> customer_id)
    df.columns = clean_cols.str.lower().str.replace(r'_+', '_', regex=True)
    return df


In [7]:
def preprocess_data(df: pd.DataFrame):
    """
    Prepares the feature preprocessing pipeline.
    """
    # 1. Clean column names: Strip whitespace and replace spaces with underscores
    df = transform_columns(df)
    # 2. Separate target from features
    # (Assuming the target column name becomes 'churn' after formatting)
    X = df.drop(columns=['churn', 'customer_id'], errors='ignore')
    y = df['churn']

    # 3. Identify feature types automatically or explicitly
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

    # 4. Define transformations per data type (Best Practice)
    # Numerical: Impute missing values with median, then scale
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Categorical: Impute missing values with most frequent string, then One-Hot Encode
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # 5. Bundle transformers into a ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='drop' # Explicitly drop columns we didn't specify (like IDs)
    )

    # 6. Train/Test Split BEFORE fitting anything to avoid data leakage
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42
    )

    return X_train, X_test, y_train, y_test, preprocessor

In [8]:
if __name__ == "__main__":
    # Example execution within your Docker container volume path
    data_descriptions_df = load_data('/Users/annamorgiel/code/soodabeh/retention-agent/raw_data/data_descriptions.csv')
    data_df = load_data('/Users/annamorgiel/code/soodabeh/retention-agent/raw_data/train.csv')
    #data_test_df = load_data('/Users/annamorgiel/code/soodabeh/retention-agent/raw_data/test.csv')

    print(f"data_descriptions_df shape: {data_descriptions_df.shape}")
    print(f"data_df shape: {data_df.shape}")
    print(f"")

    X_train, X_test, y_train, y_test, preprocessor  = preprocess_data(data_df)

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_test shape: {y_test.shape}")

data_descriptions_df shape: (21, 4)
data_df shape: (243787, 21)

X_train shape: (195029, 19)
y_train shape: (195029,)

X_test shape: (48758, 19)
y_test shape: (48758,)


In [9]:
data_descriptions_df

,Column_name,Column_type,Data_type,Description
0,AccountAge,Feature,integer,The age of the user's account in months.
1,MonthlyCharges,Feature,float,The amount charged to the user on a monthly ba...
2,TotalCharges,Feature,float,The total charges incurred by the user over th...
3,SubscriptionType,Feature,object,The type of subscription chosen by the user (B...
4,PaymentMethod,Feature,string,The method of payment used by the user.
5,PaperlessBilling,Feature,string,Indicates whether the user has opted for paper...
6,ContentType,Feature,string,The type of content preferred by the user (Mov...
7,MultiDeviceAccess,Feature,string,Indicates whether the user has access to the s...
8,DeviceRegistered,Feature,string,"The type of device registered by the user (TV,..."
9,ViewingHoursPerWeek,Feature,float,The number of hours the user spends watching c...


In [10]:
X_train.columns

Index(['account_age', 'monthly_charges', 'total_charges', 'subscription_type',
       'payment_method', 'paperless_billing', 'content_type',
       'multi_device_access', 'device_registered', 'viewing_hours_per_week',
       'average_viewing_duration', 'content_downloads_per_month',
       'genre_preference', 'user_rating', 'support_tickets_per_month',
       'gender', 'watchlist_size', 'parental_control', 'subtitles_enabled'],
      dtype='object')

In [11]:
X_train.head(4)

,account_age,monthly_charges,total_charges,subscription_type,payment_method,paperless_billing,content_type,multi_device_access,device_registered,viewing_hours_per_week,average_viewing_duration,content_downloads_per_month,genre_preference,user_rating,support_tickets_per_month,gender,watchlist_size,parental_control,subtitles_enabled
112198,54,18.667225,1008.030146,Premium,Credit card,No,TV Shows,Yes,Tablet,17.892616,171.307716,8,Action,1.585760,4,Female,17,Yes,Yes
35340,37,13.340015,493.580571,Premium,Electronic check,Yes,TV Shows,Yes,TV,36.133729,174.965535,35,Action,3.799440,1,Male,15,No,Yes
33223,77,11.440717,880.935206,Premium,Credit card,Yes,TV Shows,Yes,Tablet,39.479837,66.082295,23,Fantasy,1.881155,3,Male,13,Yes,Yes
188717,13,16.872436,219.341666,Basic,Mailed check,Yes,TV Shows,Yes,Tablet,29.012386,56.929637,13,Comedy,4.642076,4,Male,13,No,No


In [12]:
X_train.value_counts()

account_age  monthly_charges  total_charges  subscription_type  payment_method    paperless_billing  content_type  multi_device_access  device_registered  viewing_hours_per_week  average_viewing_duration  content_downloads_per_month  genre_preference  user_rating  support_tickets_per_month  gender  watchlist_size  parental_control  subtitles_enabled
1            4.991154         4.991154       Premium            Credit card       No                 Both          Yes                  Mobile             1.559813                72.489247                 20                           Drama             2.860004     7                          Male    4               No                Yes                  1
80           11.062501        885.000083     Premium            Electronic check  Yes                TV Shows      No                   Tablet             18.922207               64.054193                 8                            Fantasy           2.843251     2                         

In [13]:
X_train.describe()

,account_age,monthly_charges,total_charges,viewing_hours_per_week,average_viewing_duration,content_downloads_per_month,user_rating,support_tickets_per_month,watchlist_size
count,195029.000000,195029.000000,195029.000000,195029.000000,195029.000000,195029.000000,195029.000000,195029.000000,195029.000000
mean,60.012885,12.491444,750.074857,20.522810,92.307832,24.494726,3.003829,4.503781,12.018582
std,34.281401,4.329012,523.050523,11.237995,50.499969,14.428398,1.155021,2.870218,7.194916
min,1.000000,4.990112,4.991154,1.000065,5.000937,0.000000,1.000007,0.000000,0.000000
25%,30.000000,8.737663,328.474207,10.790555,48.475785,12.000000,2.003225,2.000000,6.000000
50%,60.000000,12.496669,649.257275,20.574123,92.314366,24.000000,3.001909,4.000000,12.000000
75%,90.000000,16.235997,1088.911209,30.244590,135.916057,37.000000,4.004423,7.000000,18.000000
max,119.000000,19.989957,2378.723844,39.999723,179.999275,49.000000,4.999973,9.000000,24.000000


In [14]:
X_train.isnull().sum()

account_age                    0
monthly_charges                0
total_charges                  0
subscription_type              0
payment_method                 0
paperless_billing              0
content_type                   0
multi_device_access            0
device_registered              0
viewing_hours_per_week         0
average_viewing_duration       0
content_downloads_per_month    0
genre_preference               0
user_rating                    0
support_tickets_per_month      0
gender                         0
watchlist_size                 0
parental_control               0
subtitles_enabled              0
dtype: int64